# NaturalisticDiffInt — 03: Brain Comparison

**Goal:** Compare NMPH model RSA change matrices with preprocessed fMRI data from StudyForrest:
1. Load model RSA results from notebook 02 (from Google Drive)
2. Load preprocessed fMRI BOLD timeseries (Schaefer 400 parcels, TR-aligned)
3. Compute fMRI RSA matrices per parcel, per run
4. Correlate model RSA change with brain RSA per parcel (Spearman r)
5. Map correlations back to parcels; identify hippocampal, DMN, TPN parcels
6. Test H1 (differentiation → hippocampal RSA decrease) and H4 (global state modulation)

**Expected Google Drive layout:**
```
MyDrive/NaturalisticDiffInt/
    neural/
        bold_schaefer400_run{1-8}.npy     # (n_trs, 400) parcellated BOLD, z-scored
        global_state_run{1-8}.npy         # (n_trs,) TPN dominance score [optional, for H4]
    results/
        nmph_naturalistic_results.pkl     # output from notebook 02
```
The Schaefer 400-parcel atlas (Yeo 17-network labels) is used for network-level averaging.


## 0. GitHub Sync — Setup

In [ ]:
from google.colab import userdata
import os, subprocess

GITHUB_USER  = "drgzkr"
GITHUB_REPO  = "NaturalisticDiffInt"
REPO_PATH    = f"/content/{GITHUB_REPO}"
NOTEBOOK_REL = "notebooks/analysis/03_brain_comparison.ipynb"

_token  = userdata.get("GITHUB_TOKEN")
_remote = f"https://{_token}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
subprocess.run(["git", "config", "--global", "user.name", "Colab"], check=True)
subprocess.run(["git", "config", "--global", "user.email", "colab@naturalistic-diffint.local"], check=True)

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", _remote, REPO_PATH], check=True)
    print(f"Cloned -> {REPO_PATH}")
else:
    subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", _remote])
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
    print(f"Pulled -> {REPO_PATH}")
del _token, _remote
print(f"Ready: {REPO_PATH}")


## 1. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT   = "/content/drive/MyDrive/NaturalisticDiffInt"
NEURAL_DIR   = f"{DRIVE_ROOT}/neural"
RESULTS_DIR  = f"{DRIVE_ROOT}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

RUNS     = list(range(1, 9))
N_PARCEL = 400    # Schaefer 400

# Check files
for run in RUNS:
    fp = f"{NEURAL_DIR}/bold_schaefer400_run{run}.npy"
    print(f"Run {run} BOLD: {'OK' if os.path.exists(fp) else 'MISSING'}")

res_path = f"{RESULTS_DIR}/nmph_naturalistic_results.pkl"
print(f"Model results: {'OK' if os.path.exists(res_path) else 'MISSING — run notebook 02 first'}")


## 2. Imports

In [ ]:
import sys
sys.path.insert(0, REPO_PATH)

import pickle, warnings
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
from scipy.stats import spearmanr, pearsonr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
np.random.seed(42)

print("Imports OK.")


## 3. Load model results

In [ ]:
with open(res_path, 'rb') as f:
    model_pkg = pickle.load(f)

rsa_before = model_pkg['rsa_before']   # {run: (n_events, n_events)}
rsa_after  = model_pkg['rsa_after']
rsa_change = model_pkg['rsa_change']
onsets     = model_pkg['onsets']       # {run: (n_events,)}
durations  = model_pkg['durations']    # {run: (n_events,)}
n_events   = model_pkg['n_events']
cfg        = model_pkg['config']
model_log  = model_pkg['log']          # {run: list of episode dicts}

print("Model config:", cfg)
for run in RUNS:
    print(f"  Run {run}: {n_events[run]} events, "
          f"RSA shape: {rsa_change[run].shape}")


In [ ]:
# ── Auto-print: model configuration recap ───────────────────────────────────
print("=" * 60)
print("MODEL RESULTS LOADED")
print("=" * 60)
print(f"  OSC_AMP             : {cfg['osc_amp']}")
print(f"  N_CATEGORY          : {cfg['n_category']}")
print(f"  COMPETITOR_THRESH   : {cfg['competitor_threshold']}")
print(f"  MIN_TEMPORAL_GAP    : {cfg['min_temporal_gap']}")
print(f"  N_HIDDEN            : {cfg['n_hidden']}")
print()
all_log = [e for r in RUNS for e in model_log[r]]
if all_log:
    all_deltas_model = [e['delta_sim'] for e in all_log]
    dirs = [e['direction'] for e in all_log]
    print(f"  Total competition episodes : {len(all_log)}")
    print(f"  Differentiation : {dirs.count('differentiation')} ({dirs.count('differentiation')/len(dirs)*100:.1f}%)")
    print(f"  Integration     : {dirs.count('integration')} ({dirs.count('integration')/len(dirs)*100:.1f}%)")
    print(f"  Mean Δsim       : {np.mean(all_deltas_model):+.4f}")
    dom = 'differentiation' if dirs.count('differentiation') > dirs.count('integration') else 'integration'
    print(f"  → Dominant outcome from notebook 02: {dom}")
print("=" * 60)


**Before continuing:** Confirm that the model configuration above matches the settings used
in notebook 02. If you re-ran notebook 02 with different hyperparameters (e.g. a different
`osc_amp` or `competitor_threshold`), make sure to re-save the `.pkl` and reload here.

The key parameter to note is `osc_amp` — this determines whether the model was operating in a
differentiation or integration regime. The brain comparisons below are only interpretable in
relation to this: if the model mostly differentiated, we expect negative correlations between
model RSA change and hippocampal RSA change (H1); if it mostly integrated, we expect positive.


## 4. Load BOLD data and compute fMRI RSA

In [ ]:
HRF_DELAY = 4   # TRs; approximate HRF peak delay

def load_bold(run):
    """Load Schaefer 400 parcellated BOLD. Expected: (n_trs, 400), z-scored."""
    return np.load(f"{NEURAL_DIR}/bold_schaefer400_run{run}.npy").astype(np.float32)

def event_avg_bold(bold, event_onsets, event_durations):
    """Mean BOLD in each event window, HRF-corrected."""
    n_ev = len(event_onsets)
    ev   = np.zeros((n_ev, bold.shape[1]), dtype=np.float32)
    for i, (s, d) in enumerate(zip(event_onsets, event_durations)):
        start = int(s) + HRF_DELAY
        end   = min(int(s + d) + HRF_DELAY, bold.shape[0])
        start = max(0, min(start, bold.shape[0]-1))
        ev[i] = bold[start:end].mean(0) if end > start else bold[start]
    return ev

def pearson_rsa(event_reps):
    """Pairwise Pearson r across events. Shape: (n_events, n_events)."""
    return np.corrcoef(event_reps)

print("Computing fMRI RSA matrices...")
fmri_rsa = {}   # {run: (n_events, n_events, n_parcels)}

for run in RUNS:
    bold = load_bold(run)
    print(f"  Run {run}: BOLD {bold.shape}", end=" ")
    ev_bold = event_avg_bold(bold, onsets[run], durations[run])   # (n_ev, 400)
    n_ev    = n_events[run]

    # Per-parcel RSA: (n_events × n_events) for each of 400 parcels
    rsa_parcels = np.zeros((n_ev, n_ev, N_PARCEL), dtype=np.float32)
    for p in range(N_PARCEL):
        col = ev_bold[:, p:p+1]   # (n_ev, 1)
        rsa_parcels[:, :, p] = np.corrcoef(col.ravel().reshape(-1, 1).T
                                             + np.zeros((n_ev, n_ev))) if n_ev > 1                                  else np.ones((n_ev, n_ev))

    # Cleaner: use all parcels at once for event RSA (event × event Pearson)
    # Each parcel is a single feature — but we want the correlation across parcels per event
    # Standard approach: event RSA = Pearson r of BOLD patterns (across all parcels)
    rsa_full = pearson_rsa(ev_bold)   # (n_ev, n_ev) — main RSA matrix
    fmri_rsa[run] = dict(full=rsa_full, ev_bold=ev_bold)
    print(f"-> RSA {rsa_full.shape}")

print("Done.")


**Note on the fMRI RSA approach used here:** We compute event-level RSA matrices by
averaging BOLD signal within each event window (HRF-corrected by `HRF_DELAY` TRs),
then computing Pearson r across all 400 Schaefer parcels. This gives one RSA matrix
per run that captures the full multivariate similarity structure of brain activity.

**Cross-run RSA change** (run 8 − run 1) approximates what happens to representational
geometry over repeated exposure to the same film. This is the closest naturalistic analogue
to the pre/post RSA measurements in the original lab paradigms (Ritvo et al. simulate
Chanales 2021 which also uses repeated exposure to measure representational change).

**Important caveats:**
- StudyForrest participants watched the film once; runs 1–8 are different segments of one
  continuous exposure, not repeated viewings. Cross-run RSA change therefore conflates
  within-film representational dynamics with any genuine learning/memory consolidation effects.
- A stronger design would use a repeated-viewing dataset. For now, treat cross-run RSA change
  as a proxy for representational dynamics over time.


## 5. Parcel-level model–brain RSA correlation

For each parcel: correlate model RSA change vector (upper triangle) with fMRI RSA change vector.

In [ ]:
def upper_tri(mat):
    """Upper triangle (above diagonal), flattened."""
    n = mat.shape[0]
    idx = np.triu_indices(n, k=1)
    return mat[idx]

def compute_fmri_rsa_change(bold_run_early, bold_run_late, onsets_e, dur_e, onsets_l, dur_l):
    """
    Cross-run RSA change: RSA(run 5-8) - RSA(run 1-4).
    Approximates 'after learning' vs 'before learning' in a repeated-exposure context.
    """
    ev_early = event_avg_bold(bold_run_early, onsets_e, dur_e)
    ev_late  = event_avg_bold(bold_run_late,  onsets_l, dur_l)
    # Align events (use minimum n_events across runs)
    n = min(ev_early.shape[0], ev_late.shape[0])
    return pearson_rsa(ev_late[:n]) - pearson_rsa(ev_early[:n])

# Aggregate runs: early = runs 1-4, late = runs 5-8
# Use run 1 & 8 as before/after for maximum temporal gap
print("Computing model–brain RSA correlations per parcel...")

bold_r1 = load_bold(1); bold_r8 = load_bold(8)
n_common = min(n_events[1], n_events[8])

# fMRI RSA in each parcel: run 8 minus run 1 (cross-run RSA change)
fmri_change_parcel = np.zeros((n_common, n_common, N_PARCEL), dtype=np.float32)
for p in range(N_PARCEL):
    col1 = event_avg_bold(bold_r1[:, :], onsets[1], durations[1])[:n_common, p]
    col8 = event_avg_bold(bold_r8[:, :], onsets[8], durations[8])[:n_common, p]
    # Trick: RSA for single parcel = outer product of z-scored activation
    c1 = (col1 - col1.mean()) / (col1.std() + 1e-8)
    c8 = (col8 - col8.mean()) / (col8.std() + 1e-8)
    rsa1 = np.outer(c1, c1)
    rsa8 = np.outer(c8, c8)
    fmri_change_parcel[:, :, p] = rsa8 - rsa1

# Model RSA change (using run 1 model results, truncated to n_common)
model_change_vec = upper_tri(rsa_change[1][:n_common, :n_common])

# Spearman r per parcel
r_map = np.zeros(N_PARCEL)
p_map = np.zeros(N_PARCEL)
for p in range(N_PARCEL):
    brain_vec = upper_tri(fmri_change_parcel[:, :, p])
    if brain_vec.std() < 1e-6:
        r_map[p] = np.nan; p_map[p] = np.nan
    else:
        r, pv = spearmanr(model_change_vec, brain_vec)
        r_map[p] = r; p_map[p] = pv

# FDR correction
valid = ~np.isnan(p_map)
_, p_fdr, _, _ = multipletests(p_map[valid], method='fdr_bh')
p_map_fdr = np.full(N_PARCEL, np.nan)
p_map_fdr[valid] = p_fdr

sig_parcels = np.where((p_map_fdr < 0.05) & valid)[0]
print(f"Significant parcels (FDR q<0.05): {len(sig_parcels)} / {N_PARCEL}")
print(f"Max r = {np.nanmax(r_map):.3f}  Min r = {np.nanmin(r_map):.3f}")


In [ ]:
# ── Auto-print: parcel RSA correlation summary ──────────────────────────────
valid_mask = ~np.isnan(r_map)
r_valid = r_map[valid_mask]

print("=" * 65)
print("PARCEL-LEVEL MODEL–BRAIN RSA CORRELATION")
print("=" * 65)
print(f"  Parcels computed         : {valid_mask.sum()} / {len(r_map)}")
print(f"  Significant (FDR q<0.05): {len(sig_parcels)}")
print(f"  Mean r (all parcels)     : {r_valid.mean():+.4f}  (SD={r_valid.std():.4f})")
print(f"  Max r                    : {r_valid.max():+.4f}  (parcel {np.nanargmax(r_map)})")
print(f"  Min r                    : {r_valid.min():+.4f}  (parcel {np.nanargmin(r_map)})")
print(f"  Parcels with r > 0       : {(r_valid>0).sum()} ({(r_valid>0).mean()*100:.1f}%)")
print(f"  Parcels with r < 0       : {(r_valid<0).sum()} ({(r_valid<0).mean()*100:.1f}%)")
print()

if len(sig_parcels) == 0:
    print("⚠  No significant parcels after FDR correction.")
    print("   Possible reasons:")
    print("   1. Insufficient competition episodes — check model results above.")
    print("   2. osc_amp not in differentiation zone — check toy validation (NB01).")
    print("   3. Cross-run RSA change too noisy — consider averaging more runs.")
    print("   4. HRF_DELAY may need adjustment for your preprocessing.")
elif len(sig_parcels) < 10:
    print(f"ℹ  {len(sig_parcels)} significant parcels — sparse but present.")
    print("   Check whether significant parcels cluster in hippocampus/DMN (expected).")
else:
    print(f"✓  {len(sig_parcels)} significant parcels detected.")

# Direction check relative to model dominance
dom_direction_model = 'differentiation' if dirs.count('differentiation') > dirs.count('integration') else 'integration'
mean_r = r_valid.mean()
if dom_direction_model == 'differentiation' and mean_r < 0:
    print()
    print("✓  Direction consistent with H1: model differentiation correlates with")
    print("   negative brain RSA change (reduced hippocampal-area pattern similarity).")
elif dom_direction_model == 'integration' and mean_r > 0:
    print()
    print("✓  Direction consistent with H2: model integration correlates with")
    print("   positive brain RSA change (increased pattern similarity).")
else:
    print()
    print("ℹ  Direction of mean r does not straightforwardly match the dominant model")
    print("   outcome. This may reflect regional variation (some areas differentiate,")
    print("   some integrate) — inspect the network-level plot in the next section.")
print("=" * 65)


**How to read the parcel r-map:** The Spearman r value per parcel measures whether the *pattern* of
representational change predicted by the NMPH model (which event pairs became more/less similar)
matches the *pattern* of RSA change observed in that brain region.

- **r > 0**: where the model predicts integration (similarity increase), that parcel shows
  increased RSA; where the model predicts differentiation, that parcel shows decreased RSA.
  This is the "model tracks brain" result — the model's NMPH dynamics match the regional
  representational geometry.
- **r < 0**: the opposite — this region differentiates where the model integrates, and vice versa.
  This could reflect that the region operates in a different inhibitory regime than the model
  (different effective osc_amp), consistent with the TRW gradient hypothesis (H5).
- **r ≈ 0**: the model's competition structure does not explain RSA change in this region —
  either because it doesn't participate in memory competition, or because the feature space
  driving competition doesn't capture the signals relevant to that region.

**Significance caveat:** With 400 parcels and noisy cross-run RSA change, the power per parcel
is limited. The FDR-corrected significant parcels should be treated as directional candidates
for targeted follow-up, not definitive conclusions.


## 6. Parcel r-map visualisation

In [ ]:
# Parcel bar plot sorted by r-value
sorted_idx = np.argsort(r_map)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
colors = ['firebrick' if r > 0 else 'steelblue' for r in r_map[sorted_idx[:50]]]
ax.bar(range(50), r_map[sorted_idx[:50]], color=colors, alpha=0.8)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel("Parcel rank (top 50)")
ax.set_ylabel("Spearman r (model change ~ brain RSA change)")
ax.set_title("Top 50 parcels: model–brain RSA correlation")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[1]
ax.hist(r_map[valid], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='k', lw=0.8, ls='--')
mn = np.nanmean(r_map)
ax.axvline(mn, color='red', lw=1.5, label=f"Mean r = {mn:.3f}")
ax.set_xlabel("Spearman r"); ax.set_ylabel("N parcels")
ax.set_title("Distribution of model–brain RSA correlations")
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/model_brain_r_map.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. Network-level analysis

Average r-values within Yeo 17 networks (using Schaefer parcel → network labels).

In [ ]:
# Download Schaefer 400 → Yeo 17 network labels from nilearn
!pip install nilearn -q

from nilearn.datasets import fetch_atlas_schaefer_2018
atlas = fetch_atlas_schaefer_2018(n_rois=400, yeo_networks=17, resolution_mm=2)
labels = [l.decode() if isinstance(l, bytes) else l for l in atlas.labels]

# Extract network name from label (e.g. '17Networks_LH_VisCent_ExStr_1' → 'VisCent_ExStr')
def extract_network(label):
    parts = label.split('_')
    if len(parts) >= 4:
        return '_'.join(parts[3:-1])  # network name without hemisphere and parcel number
    return label

network_labels = [extract_network(l) for l in labels]
unique_networks = sorted(set(network_labels))
print(f"Found {len(unique_networks)} unique networks")

# Average r per network
network_r = {}
for net in unique_networks:
    parcel_mask = np.array([n == net for n in network_labels])
    network_r[net] = np.nanmean(r_map[parcel_mask])

# Sort by r
sorted_nets = sorted(network_r.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(14, 5))
net_names, net_rs = zip(*sorted_nets)
colors = ['firebrick' if r > 0 else 'steelblue' for r in net_rs]
ax.bar(range(len(net_names)), net_rs, color=colors, alpha=0.8)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xticks(range(len(net_names)))
ax.set_xticklabels(net_names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Mean Spearman r")
ax.set_title("Model–brain RSA correlation by Yeo 17 network")
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/network_r_map.png", dpi=150, bbox_inches='tight')
plt.show()

# Highlight hippocampus-adjacent networks
hipp_keywords = ['Para', 'Default', 'DMN', 'Hipp']
print("\nNetworks with potential hippocampal involvement:")
for name, r in sorted_nets:
    if any(k.lower() in name.lower() for k in hipp_keywords):
        print(f"  {name:40s}  r = {r:+.4f}")


In [ ]:
# ── Auto-print: network ranking ─────────────────────────────────────────────
print("=" * 65)
print("NETWORK-LEVEL RSA CORRELATION RANKING")
print("=" * 65)
print(f"  {'Network':40s}  {'Mean r':>8}  {'Direction':>15}")
print("  " + "-"*63)
for name, r in sorted_nets:
    direction = ("↑ integration" if r > 0.05 else
                 "↓ differentiation" if r < -0.05 else "~  no preference")
    marker = "★" if any(k.lower() in name.lower()
                        for k in ['Para','Default','Limbic','Hipp']) else " "
    print(f"  {marker} {name:38s}  {r:>+8.4f}  {direction:>15}")
print()
print("  ★ = hippocampus-adjacent or default-mode network")
print()

top3    = sorted_nets[:3]
bottom3 = sorted_nets[-3:]
print(f"  Top 3 (model-correlated)   : {', '.join(n for n,_ in top3)}")
print(f"  Bottom 3 (anti-correlated) : {', '.join(n for n,_ in bottom3)}")
print()

hipp_nets = [(n,r) for n,r in sorted_nets
             if any(k.lower() in n.lower() for k in ['Para','Default','Limbic'])]
if hipp_nets:
    hipp_r_mean = np.mean([r for _,r in hipp_nets])
    print(f"  Hippocampus-adjacent/DMN networks mean r: {hipp_r_mean:+.4f}")
    if hipp_r_mean < -0.02:
        print("  → Consistent with H1: DMN/hippocampal areas show RSA change in the")
        print("    opposite direction to model prediction (differentiation).")
    elif hipp_r_mean > 0.02:
        print("  → Consistent with H2: DMN/hippocampal areas show RSA change in the")
        print("    same direction as model prediction (integration).")
    else:
        print("  → No clear hippocampal preference — may need higher osc_amp or more events.")
print("=" * 65)


**Interpreting the network ranking:** The Yeo 17 networks provide a coarser but more robust
summary than individual parcels. Each network value is the mean Spearman r across its constituent
Schaefer 400 parcels.

**What the TRW hierarchy predicts (H5):** If the oscillation amplitude gradient maps onto the
temporal receptive window (TRW) hierarchy, then:
- **Visual networks** (short TRW, low inhibitory tone) → high competitor reactivation → *integration*
  → positive r if model predominantly integrates
- **Default mode / hippocampal networks** (long TRW, high inhibitory tone) → moderate competitor
  reactivation → *differentiation* → negative r (opposite direction to model)

This would produce a systematic rank ordering of networks from integration-positive to
differentiation-positive along the sensory-to-transmodal axis.

**If the ranking is flat (all r ≈ 0):** The model's event pairmate structure may not be capturing
the right level of representational competition. Consider trying alternative feature spaces
(e.g. language model embeddings as category features) or a finer/coarser competitor threshold.


## 8. H4: Global brain state modulation (TPN vs DMN dominance)

If global state timeseries are available, test whether events occurring during TPN-dominant states show more differentiation.

In [ ]:
# Check if global state data is available
gs_path = f"{NEURAL_DIR}/global_state_run1.npy"
HAS_GLOBAL_STATE = os.path.exists(gs_path)
print(f"Global state data: {'available' if HAS_GLOBAL_STATE else 'not found — skipping H4'}")

if HAS_GLOBAL_STATE:
    # Load TPN dominance scores for each run
    def load_gs(run):
        return np.load(f"{NEURAL_DIR}/global_state_run{run}.npy").astype(np.float32)

    # Assign each event a mean global state value
    run_event_gs = {}
    for run in RUNS:
        gs = load_gs(run)
        ev_gs = np.array([
            gs[int(s):int(s+d)].mean()
            for s, d in zip(onsets[run], durations[run])
        ])
        run_event_gs[run] = ev_gs

    # For each competition episode: extract global state of target event
    # and test whether high-TPN events → more differentiation
    episode_gs    = []
    episode_delta = []
    for run in RUNS:
        for ep in model_log[run]:
            target_idx = ep['target_idx']
            if target_idx < len(run_event_gs[run]):
                episode_gs.append(run_event_gs[run][target_idx])
                episode_delta.append(ep['delta_sim'])

    episode_gs    = np.array(episode_gs)
    episode_delta = np.array(episode_delta)

    r_gs, p_gs = spearmanr(episode_gs, episode_delta)
    print(f"\nH4 test: TPN dominance × Δsim  r = {r_gs:.3f}, p = {p_gs:.4f}")

    # Median split
    med = np.median(episode_gs)
    tpn_high = episode_delta[episode_gs >= med]
    tpn_low  = episode_delta[episode_gs <  med]
    t_stat, p_ttest = stats.mannwhitneyu(tpn_high, tpn_low, alternative='less')
    print(f"H4 Mann-Whitney (TPN-high < TPN-low Δsim): U={t_stat:.1f}, p={p_ttest:.4f}")
    print(f"  TPN-high mean Δsim = {tpn_high.mean():+.4f}")
    print(f"  TPN-low  mean Δsim = {tpn_low.mean():+.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ax = axes[0]
    ax.scatter(episode_gs, episode_delta, alpha=0.3, s=15, color='steelblue')
    z = np.polyfit(episode_gs, episode_delta, 1)
    xs = np.linspace(episode_gs.min(), episode_gs.max(), 100)
    ax.plot(xs, np.polyval(z, xs), 'r-', lw=2, label=f'r={r_gs:.3f}, p={p_gs:.3f}')
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_xlabel("TPN dominance score at event")
    ax.set_ylabel("Δ cosine similarity (NMPH competition)")
    ax.set_title("H4: Global brain state × representational change")
    ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    ax = axes[1]
    ax.violinplot([tpn_low, tpn_high], positions=[0, 1], showmedians=True)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['TPN-low (DMN)', 'TPN-high (TPN)'])
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_ylabel("Δ cosine similarity")
    ax.set_title(f"Median split  (p={p_ttest:.4f})")
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/H4_global_state.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("\nTo test H4, provide global_state_run{1-8}.npy in the neural/ folder.")
    print("These can be computed from the FlippingPaper TPN/DMN state timeseries.")


**Interpreting H4 (global brain state modulation):**

The NMPH model predicts that oscillation amplitude — the key determinant of competitor reactivation
level — is modulated by inhibitory tone, which in turn is modulated by global brain state (TPN/DMN
dominance). Specifically:

- **TPN-dominant states** (task-positive network active) reflect higher cortical arousal and stronger
  inhibitory surround. This should *limit* competitor reactivation, placing more events in the
  moderate-activity (differentiation) zone → negative Δsim
- **DMN-dominant states** (default mode active) reflect a more internally-oriented, less
  inhibition-constrained regime. Competitors may reactivate more strongly, pushing towards
  integration → positive Δsim

**What to look for:**
- The scatter plot should show a positive trend (higher TPN → more negative Δsim, i.e. more
  differentiation) — but note this will be visible as a **negative slope** if TPN dominance is
  coded as positive values (higher = more TPN)
- The median split violin plot should show TPN-high events with more negative Δsim than TPN-low
- A significant Mann-Whitney U (p < 0.05) would provide the first direct link between global
  brain state and NMPH competition direction in naturalistic data

**If not significant:** The effect may be present but underpowered with a single film. Pooling
across subjects would be the natural next step.


## 9. H1: Hippocampal RSA decrease for differentiated pairs

Test whether event pairs the model differentiates show reduced hippocampal RSA (run 1 → run 8).

In [ ]:
# Define hippocampal parcels: look for 'Default' or 'Para' in Schaefer labels
# (Schaefer 400 doesn't have hippocampus directly — use DMN parcels as proxy,
#  or if you have a separate hippocampal mask, load it here)
hipp_parcel_mask = np.array([
    any(k in l for k in ['Default', 'Para', 'Limbic']) for l in network_labels
])
hipp_parcel_idx = np.where(hipp_parcel_mask)[0]
print(f"Hippocampus-adjacent parcels: {hipp_parcel_idx.sum()}")

# RSA change in those parcels for differentiated vs non-differentiated pairs
n_common = min(n_events[1], n_events[8])
model_diff_matrix = rsa_change[1][:n_common, :n_common]  # (n_ev, n_ev)

# For each event pair: model predicts diff (<0) or not
# Compare brain RSA change for those pairs
bold_ev_r1 = event_avg_bold(bold_r1, onsets[1], durations[1])[:n_common]
bold_ev_r8 = event_avg_bold(bold_r8, onsets[8], durations[8])[:n_common]

# RSA in hippocampal parcels
hipp_r1 = pearson_rsa(bold_ev_r1[:, hipp_parcel_idx])
hipp_r8 = pearson_rsa(bold_ev_r8[:, hipp_parcel_idx])
hipp_change = hipp_r8 - hipp_r1

# Upper triangle pairs
pairs_idx = np.triu_indices(n_common, k=1)
model_change_flat = model_diff_matrix[pairs_idx]
hipp_change_flat  = hipp_change[pairs_idx]

# Separate differentiated vs integrated pairs
is_diff = model_change_flat < -0.01
is_intg = model_change_flat >  0.01

print(f"Pairs classified as differentiation: {is_diff.sum()}")
print(f"Pairs classified as integration: {is_intg.sum()}")

if is_diff.sum() > 5 and is_intg.sum() > 5:
    t, p = stats.mannwhitneyu(hipp_change_flat[is_diff], hipp_change_flat[is_intg])
    print(f"\nH1: Hippocampal RSA change  diff pairs={hipp_change_flat[is_diff].mean():+.4f}"
          f"  intg pairs={hipp_change_flat[is_intg].mean():+.4f}")
    print(f"     Mann-Whitney U={t:.1f}, p={p:.4f}")

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.violinplot([hipp_change_flat[is_diff], hipp_change_flat[is_intg]],
                  positions=[0, 1], showmedians=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Model: Differentiation', 'Model: Integration'])
    ax.axhline(0, color='k', lw=0.8, ls='--')
    ax.set_ylabel("Hippocampal RSA change (run 8 − run 1)")
    ax.set_title(f"H1: Do differentiated pairs show reduced hippocampal RSA?\n"
                 f"Mann-Whitney p = {p:.4f}")
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/H1_hippocampal_rsa.png", dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough pairs in each category — try adjusting COMPETITOR_THRESH in notebook 02.")


In [ ]:
# ── Auto-print: H1 result interpretation ────────────────────────────────────
if 'is_diff' in dir() and is_diff.sum() > 5 and is_intg.sum() > 5:
    diff_mean = float(hipp_change_flat[is_diff].mean())
    intg_mean = float(hipp_change_flat[is_intg].mean())
    print("=" * 65)
    print("H1: HIPPOCAMPAL RSA DECREASE FOR DIFFERENTIATED PAIRS")
    print("=" * 65)
    print(f"  Model-differentiated pairs: hippocampal RSA change = {diff_mean:+.4f}")
    print(f"  Model-integrated pairs    : hippocampal RSA change = {intg_mean:+.4f}")
    print(f"  Difference                : {diff_mean - intg_mean:+.4f}")
    print(f"  Mann-Whitney p            : {p:.4f}")
    print()
    if p < 0.05 and diff_mean < intg_mean:
        print("✓  H1 SUPPORTED: differentiated pairs show reduced hippocampal RSA,")
        print("   integrated pairs show increased hippocampal RSA (or less decrease).")
        print("   This is the key NMPH prediction applied to naturalistic fMRI.")
    elif p < 0.05 and diff_mean > intg_mean:
        print("⚠  Significant but opposite direction — hippocampal RSA increases")
        print("   for model-differentiated pairs. This would falsify H1 as stated.")
        print("   Possible explanations: wrong HRF delay, atlas mismatch, or model")
        print("   osc_amp in integration zone despite low mean Δsim.")
    elif p >= 0.05 and abs(diff_mean - intg_mean) > 0.01:
        print("ℹ  Trend in predicted direction but not significant.")
        print("   Power is limited with a single subject/run comparison.")
        print("   Consider cross-subject averaging or longer time windows.")
    else:
        print("ℹ  No difference between groups. The model's competition structure")
        print("   may not map onto hippocampal RSA change for this feature space.")
    print()
    print("  Note: 'hippocampal' here means Default/Para/Limbic Schaefer parcels.")
    print("  For a stricter test, provide a dedicated hippocampal subfield mask.")
    print("=" * 65)


**Interpreting H1:** This is the central empirical prediction of the project. The NMPH framework
(Ritvo et al. 2024) predicts that when two event traces undergo differentiation — their hidden
representations are driven apart by weakening of shared connections — the brain region that is
the source of this differentiation should show *decreased* pattern similarity (RSA) between those
events over time.

**Limitations of the current operationalisation:**
1. We use "Default/Para/Limbic" Schaefer parcels as a proxy for hippocampus. A dedicated
   hippocampal mask (e.g. from a manual ROI or the Julich-Brain atlas via `siibra`) would give
   a sharper test.
2. Cross-run RSA change (run 8 − run 1) is a noisy proxy for representational change — it
   conflates genuine learning-driven change with position-in-film effects.
3. The comparison is across event pairs within a single run, which limits statistical power.

**The cleanest test of H1** would use a repeated-viewing dataset (e.g. Sherlock, where
participants watched the film and then recalled it — allowing pre/post RSA comparison) and a
hippocampal subfield mask. That is the natural next step after this pilot.


## 10. Save summary statistics

In [ ]:
summary = dict(
    r_map=r_map, p_map=p_map, p_map_fdr=p_map_fdr,
    sig_parcels=sig_parcels.tolist(),
    network_r=network_r,
)
with open(f"{RESULTS_DIR}/brain_comparison_summary.pkl", 'wb') as f:
    pickle.dump(summary, f)
np.save(f"{RESULTS_DIR}/r_map_parcel.npy", r_map)
np.save(f"{RESULTS_DIR}/p_map_fdr.npy", p_map_fdr)
print("Summary saved.")


## 11. GitHub Sync — Push

In [ ]:
import json as _j, subprocess as _sp
from datetime import datetime as _dt
from google.colab import _message
import os as _os

_nb   = _message.blocking_request('get_ipynb', timeout_sec=30)
_dest = f"{REPO_PATH}/{NOTEBOOK_REL}"
_os.makedirs(_os.path.dirname(_dest), exist_ok=True)
with open(_dest,'w') as _f: _j.dump(_nb,_f,indent=1)
_msg = f"[colab] 03_brain_comparison: {_dt.now():%Y-%m-%d %H:%M}"
_sp.run(["git","-C",REPO_PATH,"add",NOTEBOOK_REL], check=True)
_res = _sp.run(["git","-C",REPO_PATH,"commit","-m",_msg], capture_output=True, text=True)
if "nothing to commit" in _res.stdout: print("Nothing to commit.")
else:
    _sp.run(["git","-C",REPO_PATH,"push"], check=True)
    print(f"Pushed: {_msg}")
